<a href="https://colab.research.google.com/github/MettaTan/SIT-UofG-QC-Assignment/blob/main/BB84_Attacker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 75.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=c7fca533e0aa98fb0d30e80315a00b166efb9dacdd1d2265a0ed25dbb460bd57
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

## Overview

Same protocol as the plain version, but with an attacker **Eve** sitting on the
quantum channel and performing an **intercept-resend** attack:

1. For each qubit Alice sends, Eve picks a random basis.
2. Eve measures the qubit in that basis (learning one bit).
3. Eve prepares a fresh qubit carrying the bit she measured, in her chosen basis,
   and forwards it to Bob.

Because she doesn't know Alice's basis, on average Eve picks the wrong basis half
the time. Those qubits arrive at Bob in the wrong basis, producing a ~25%
disagreement rate on the sifted key — easy to spot during the public check.

Each agent's section below is labelled **ALICE**, **EVE**, or **BOB**.


In [ ]:
simulator = BasicSimulator()


## Quantum random bit generator


In [ ]:
CHUNK = 16  # BasicSimulator's per-circuit qubit limit.

def quantum_random_bits(n):
    """Return a list of n uniformly random bits, each from measuring |+>.

    BasicSimulator caps at 24 qubits per circuit, so we batch in chunks.
    """
    bits = []
    while len(bits) < n:
        k = min(CHUNK, n - len(bits))
        qc = QuantumCircuit(k, k)
        qc.h(range(k))
        qc.measure(range(k), range(k))
        result = simulator.run(qc, shots=1).result()
        bitstring = list(result.get_counts().keys())[0]
        bits.extend(int(b) for b in bitstring[::-1])
    return bits[:n]


## Helper functions (same as the plain notebook)


In [ ]:
def prepare_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

def measure_qubit(qc, basis):
    qc = qc.copy()
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1).result()
    return int(list(result.get_counts().keys())[0])


## ALICE


In [ ]:
N = 128

# ----- ALICE -----
alice_bits   = quantum_random_bits(N)
alice_bases  = quantum_random_bits(N)
alice_qubits = [prepare_qubit(alice_bits[i], alice_bases[i]) for i in range(N)]
print(f"Alice prepared {N} qubits.")


Alice prepared 128 qubits.


## EVE — intercept-resend attack

Eve cannot just copy a qubit (no-cloning), and once she measures it she has
collapsed Alice's state. So she forwards a *new* qubit carrying the bit she saw,
prepared in her own chosen basis.


In [ ]:
# ----- EVE -----
eve_bases        = quantum_random_bits(N)
eve_bits         = []
forwarded_qubits = []

for i in range(N):
    bit = measure_qubit(alice_qubits[i], eve_bases[i])
    eve_bits.append(bit)
    forwarded_qubits.append(prepare_qubit(bit, eve_bases[i]))

print(f"Eve intercepted and forwarded {N} qubits.")


Eve intercepted and forwarded 128 qubits.


## BOB

Bob measures whatever arrives — which is now Eve's forwarded qubits, not Alice's originals.


In [ ]:
# ----- BOB -----
bob_bases = quantum_random_bits(N)
bob_bits  = [measure_qubit(forwarded_qubits[i], bob_bases[i]) for i in range(N)]
print(f"Bob measured {N} qubits.")


Bob measured 128 qubits.


## Sifting


In [ ]:
matching     = [i for i in range(N) if alice_bases[i] == bob_bases[i]]
alice_sifted = [alice_bits[i] for i in matching]
bob_sifted   = [bob_bits[i]   for i in matching]
eve_sifted   = [eve_bits[i]   for i in matching]

print(f"Sifted key length: {len(matching)} out of {N}")


Sifted key length: 72 out of 128


## Attack detection

Theory predicts a ~25% error rate when Eve does intercept-resend.
With threshold 15%, Eve should be caught on essentially every run.


In [ ]:
sample_size  = len(matching) // 2
sample_alice = alice_sifted[:sample_size]
sample_bob   = bob_sifted[:sample_size]

errors     = sum(1 for a, b in zip(sample_alice, sample_bob) if a != b)
error_rate = errors / sample_size if sample_size else 0
THRESHOLD  = 0.15

print(f"Sample size:  {sample_size}")
print(f"Errors found: {errors}")
print(f"Error rate:   {error_rate:.2%}")
print(f"Threshold:    {THRESHOLD:.0%}")

if error_rate > THRESHOLD:
    print("\n>>> EAVESDROPPING DETECTED -- key aborted.")
else:
    print("\n>>> Channel looks clean (Eve got lucky this round).")


Sample size:  36
Errors found: 11
Error rate:   30.56%
Threshold:    15%

>>> EAVESDROPPING DETECTED -- key aborted.


## How much did Eve actually learn?

Even on a lucky run where she avoids detection, Eve only knows ~75% of the sifted
bits with certainty: on positions where her basis matched Alice's she has the bit
exactly, and on the rest she has 50/50.


In [ ]:
agree = sum(1 for a, e in zip(alice_sifted, eve_sifted) if a == e)
print(f"Eve's bits agree with Alice's on {agree}/{len(alice_sifted)} "
      f"sifted positions ({agree/len(alice_sifted):.1%}).")


Eve's bits agree with Alice's on 54/72 sifted positions (75.0%).


## Repeated trials — detection rate

Run the protocol many times to confirm the threshold reliably catches Eve
and doesn't fire on a clean channel.


In [ ]:
def run_trial(N=64, threshold=0.15, with_eve=True):
    a_bits  = quantum_random_bits(N)
    a_bases = quantum_random_bits(N)
    a_qubits = [prepare_qubit(a_bits[i], a_bases[i]) for i in range(N)]

    if with_eve:
        e_bases = quantum_random_bits(N)
        fwd = []
        for i in range(N):
            b = measure_qubit(a_qubits[i], e_bases[i])
            fwd.append(prepare_qubit(b, e_bases[i]))
        channel = fwd
    else:
        channel = a_qubits

    b_bases = quantum_random_bits(N)
    b_bits  = [measure_qubit(channel[i], b_bases[i]) for i in range(N)]

    match = [i for i in range(N) if a_bases[i] == b_bases[i]]
    if not match:
        return False, 0.0
    a_sift = [a_bits[i] for i in match]
    b_sift = [b_bits[i] for i in match]
    s = len(match) // 2
    errs = sum(1 for x, y in zip(a_sift[:s], b_sift[:s]) if x != y)
    rate = errs / s if s else 0
    return rate > threshold, rate

TRIALS = 10

detected, rates = 0, []
for _ in range(TRIALS):
    d, r = run_trial(with_eve=True)
    detected += int(d); rates.append(r)
print(f"With Eve:    detected in {detected}/{TRIALS} runs, "
      f"mean error rate {sum(rates)/len(rates):.2%}")

clean_detected, clean_rates = 0, []
for _ in range(TRIALS):
    d, r = run_trial(with_eve=False)
    clean_detected += int(d); clean_rates.append(r)
print(f"Without Eve: false alarms {clean_detected}/{TRIALS} runs, "
      f"mean error rate {sum(clean_rates)/len(clean_rates):.2%}")


With Eve:    detected in 9/10 runs, mean error rate 27.92%
Without Eve: false alarms 0/10 runs, mean error rate 0.00%
